# Projeto

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
%pip install pyopengl
%pip install glfw
%pip install pyglm
%pip install numpy

  Using cached pyopengl-3.1.10-py3-none-any.whl.metadata (3.3 kB)
Using cached pyopengl-3.1.10-py3-none-any.whl (3.2 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached glfw-2.10.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38.py39.py310.py311.py312.py313.py314-none-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
Using cached glfw-2.10.0-py2.py27.py3.py30.py31.py32.py33.py34.py35.py36.py37.py38.py39.py310.py311.py312.py313.py314-none-manylinux_2_28_x86_64.whl (243 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached pyglm-2.8.3-cp312-cp312-manylinux_2_34_x86_64.whl.metadata (13 kB)
Using cached pyglm-2.8.3-cp312-cp312-manylinux_2_34_x86_64.whl (11.7 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)

In [2]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
import random

### Inicializando janela

In [3]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(700, 700, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

### Shaders

Note que agora usamos vec3, já que estamos em 3D.

In [4]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [5]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

### Requisitando slot para a GPU para nossos programas Vertex e Fragment Shaders

In [6]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)


### Associando nosso código-fonte aos slots solicitados

In [7]:
# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

### Compilando o Vertex Shader

Se há algum erro em nosso programa Vertex Shader, nosso app para por aqui.

In [8]:
# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")


### Compilando o Fragment Shader

Se há algum erro em nosso programa Fragment Shader, nosso app para por aqui.

In [9]:
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

### Associando os programas compilado ao programa principal

In [10]:
# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)


### Linkagem do programa

In [11]:
# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

### Preparando dados para enviar a GPU

Até aqui, compilamos nossos Shaders para que a GPU possa processá-los.

Por outro lado, as informações de vértices geralmente estão na CPU e devem ser transmitidas para a GPU.


#### Vértices do TeaPot

- Crédito: https://github.com/kretash/UtahTeapot/blob/master/teapot.h

In [12]:
vertices = np.zeros(1728, [("position", np.float32, 3)])
vertices['position'] = [
    (0.700000, -1.200000, 0.000000),
    (0.605600, -1.200000, -0.355700),
    (0.598800, -1.243700, -0.351700),
    (0.598800, -1.243700, -0.351700),
    (0.692100, -1.243700, 0.000000),
    (0.700000, -1.200000, 0.000000),
    (0.692100, -1.243700, 0.000000),
    (0.598800, -1.243700, -0.351700),
    (0.619600, -1.243700, -0.363900),
    (0.619600, -1.243700, -0.363900),
    (0.716200, -1.243700, 0.000000),
    (0.692100, -1.243700, 0.000000),
    (0.716200, -1.243700, 0.000000),
    (0.619600, -1.243700, -0.363900),
    (0.648900, -1.200000, -0.381100),
    (0.648900, -1.200000, -0.381100),
    (0.750000, -1.200000, 0.000000),
    (0.716200, -1.243700, 0.000000),
    (0.605600, -1.200000, -0.355700),
    (0.355700, -1.200000, -0.605600),
    (0.351700, -1.243700, -0.598800),
    (0.351700, -1.243700, -0.598800),
    (0.598800, -1.243700, -0.351700),
    (0.605600, -1.200000, -0.355700),
    (0.598800, -1.243700, -0.351700),
    (0.351700, -1.243700, -0.598800),
    (0.363900, -1.243800, -0.619600),
    (0.363900, -1.243800, -0.619600),
    (0.619600, -1.243700, -0.363900),
    (0.598800, -1.243700, -0.351700),
    (0.619600, -1.243700, -0.363900),
    (0.363900, -1.243800, -0.619600),
    (0.381100, -1.200000, -0.648900),
    (0.381100, -1.200000, -0.648900),
    (0.648900, -1.200000, -0.381100),
    (0.619600, -1.243700, -0.363900),
    (0.355700, -1.200000, -0.605600),
    (0.000000, -1.200000, -0.700000),
    (0.000000, -1.243700, -0.692100),
    (0.000000, -1.243700, -0.692100),
    (0.351700, -1.243700, -0.598800),
    (0.355700, -1.200000, -0.605600),
    (0.351700, -1.243700, -0.598800),
    (0.000000, -1.243700, -0.692100),
    (0.000000, -1.243800, -0.716200),
    (0.000000, -1.243800, -0.716200),
    (0.363900, -1.243800, -0.619600),
    (0.351700, -1.243700, -0.598800),
    (0.363900, -1.243800, -0.619600),
    (0.000000, -1.243800, -0.716200),
    (0.000000, -1.200000, -0.750000),
    (0.000000, -1.200000, -0.750000),
    (0.381100, -1.200000, -0.648900),
    (0.363900, -1.243800, -0.619600),
    (0.000000, -1.200000, -0.700000),
    (-0.375700, -1.200000, -0.605600),
    (-0.357600, -1.243700, -0.598800),
    (-0.357600, -1.243700, -0.598800),
    (0.000000, -1.243700, -0.692100),
    (0.000000, -1.200000, -0.700000),
    (0.000000, -1.243700, -0.692100),
    (-0.357600, -1.243700, -0.598800),
    (-0.364700, -1.243700, -0.619600),
    (-0.364700, -1.243700, -0.619600),
    (0.000000, -1.243800, -0.716200),
    (0.000000, -1.243700, -0.692100),
    (0.000000, -1.243800, -0.716200),
    (-0.364700, -1.243700, -0.619600),
    (-0.381100, -1.200000, -0.648900),
    (-0.381100, -1.200000, -0.648900),
    (0.000000, -1.200000, -0.750000),
    (0.000000, -1.243800, -0.716200),
    (-0.375700, -1.200000, -0.605600),
    (-0.615600, -1.200000, -0.355700),
    (-0.601800, -1.243700, -0.351700),
    (-0.601800, -1.243700, -0.351700),
    (-0.357600, -1.243700, -0.598800),
    (-0.375700, -1.200000, -0.605600),
    (-0.357600, -1.243700, -0.598800),
    (-0.601800, -1.243700, -0.351700),
    (-0.620000, -1.243700, -0.363900),
    (-0.620000, -1.243700, -0.363900),
    (-0.364700, -1.243700, -0.619600),
    (-0.357600, -1.243700, -0.598800),
    (-0.364700, -1.243700, -0.619600),
    (-0.620000, -1.243700, -0.363900),
    (-0.648900, -1.200000, -0.381100),
    (-0.648900, -1.200000, -0.381100),
    (-0.381100, -1.200000, -0.648900),
    (-0.364700, -1.243700, -0.619600),
    (-0.615600, -1.200000, -0.355700),
    (-0.700000, -1.200000, 0.000000),
    (-0.692100, -1.243700, 0.000000),
    (-0.692100, -1.243700, 0.000000),
    (-0.601800, -1.243700, -0.351700),
    (-0.615600, -1.200000, -0.355700),
    (-0.601800, -1.243700, -0.351700),
    (-0.692100, -1.243700, 0.000000),
    (-0.716200, -1.243700, 0.000000),
    (-0.716200, -1.243700, 0.000000),
    (-0.620000, -1.243700, -0.363900),
    (-0.601800, -1.243700, -0.351700),
    (-0.620000, -1.243700, -0.363900),
    (-0.716200, -1.243700, 0.000000),
    (-0.750000, -1.200000, 0.000000),
    (-0.750000, -1.200000, 0.000000),
    (-0.648900, -1.200000, -0.381100),
    (-0.620000, -1.243700, -0.363900),
    (-0.700000, -1.200000, 0.000000),
    (-0.605600, -1.200000, 0.355700),
    (-0.598800, -1.243700, 0.351700),
    (-0.598800, -1.243700, 0.351700),
    (-0.692100, -1.243700, 0.000000),
    (-0.700000, -1.200000, 0.000000),
    (-0.692100, -1.243700, 0.000000),
    (-0.598800, -1.243700, 0.351700),
    (-0.619600, -1.243700, 0.363900),
    (-0.619600, -1.243700, 0.363900),
    (-0.716200, -1.243700, 0.000000),
    (-0.692100, -1.243700, 0.000000),
    (-0.716200, -1.243700, 0.000000),
    (-0.619600, -1.243700, 0.363900),
    (-0.648900, -1.200000, 0.381100),
    (-0.648900, -1.200000, 0.381100),
    (-0.750000, -1.200000, 0.000000),
    (-0.716200, -1.243700, 0.000000),
    (-0.605600, -1.200000, 0.355700),
    (-0.355700, -1.200000, 0.605600),
    (-0.351700, -1.243700, 0.598800),
    (-0.351700, -1.243700, 0.598800),
    (-0.598800, -1.243700, 0.351700),
    (-0.605600, -1.200000, 0.355700),
    (-0.598800, -1.243700, 0.351700),
    (-0.351700, -1.243700, 0.598800),
    (-0.363900, -1.243700, 0.619600),
    (-0.363900, -1.243700, 0.619600),
    (-0.619600, -1.243700, 0.363900),
    (-0.598800, -1.243700, 0.351700),
    (-0.619600, -1.243700, 0.363900),
    (-0.363900, -1.243700, 0.619600),
    (-0.381100, -1.200000, 0.648900),
    (-0.381100, -1.200000, 0.648900),
    (-0.648900, -1.200000, 0.381100),
    (-0.619600, -1.243700, 0.363900),
    (-0.355700, -1.200000, 0.605600),
    (0.000000, -1.200000, 0.700000),
    (0.000000, -1.243700, 0.692100),
    (0.000000, -1.243700, 0.692100),
    (-0.351700, -1.243700, 0.598800),
    (-0.355700, -1.200000, 0.605600),
    (-0.351700, -1.243700, 0.598800),
    (0.000000, -1.243700, 0.692100),
    (0.000000, -1.243700, 0.716200),
    (0.000000, -1.243700, 0.716200),
    (-0.363900, -1.243700, 0.619600),
    (-0.351700, -1.243700, 0.598800),
    (-0.363900, -1.243700, 0.619600),
    (0.000000, -1.243700, 0.716200),
    (0.000000, -1.200000, 0.750000),
    (0.000000, -1.200000, 0.750000),
    (-0.381100, -1.200000, 0.648900),
    (-0.363900, -1.243700, 0.619600),
    (0.000000, -1.200000, 0.700000),
    (0.355700, -1.200000, 0.605600),
    (0.351700, -1.243700, 0.598800),
    (0.351700, -1.243700, 0.598800),
    (0.000000, -1.243700, 0.692100),
    (0.000000, -1.200000, 0.700000),
    (0.000000, -1.243700, 0.692100),
    (0.351700, -1.243700, 0.598800),
    (0.363900, -1.243700, 0.619600),
    (0.363900, -1.243700, 0.619600),
    (0.000000, -1.243700, 0.716200),
    (0.000000, -1.243700, 0.692100),
    (0.000000, -1.243700, 0.716200),
    (0.363900, -1.243700, 0.619600),
    (0.381100, -1.200000, 0.648900),
    (0.381100, -1.200000, 0.648900),
    (0.000000, -1.200000, 0.750000),
    (0.000000, -1.243700, 0.716200),
    (0.355700, -1.200000, 0.605600),
    (0.605600, -1.200000, 0.355700),
    (0.598800, -1.243700, 0.351700),
    (0.598800, -1.243700, 0.351700),
    (0.351700, -1.243700, 0.598800),
    (0.355700, -1.200000, 0.605600),
    (0.351700, -1.243700, 0.598800),
    (0.598800, -1.243700, 0.351700),
    (0.619600, -1.243700, 0.363900),
    (0.619600, -1.243700, 0.363900),
    (0.363900, -1.243700, 0.619600),
    (0.351700, -1.243700, 0.598800),
    (0.363900, -1.243700, 0.619600),
    (0.619600, -1.243700, 0.363900),
    (0.648900, -1.200000, 0.381100),
    (0.648900, -1.200000, 0.381100),
    (0.381100, -1.200000, 0.648900),
    (0.363900, -1.243700, 0.619600),
    (0.605600, -1.200000, 0.355700),
    (0.700000, -1.200000, 0.000000),
    (0.692100, -1.243700, 0.000000),
    (0.692100, -1.243700, 0.000000),
    (0.598800, -1.243700, 0.351700),
    (0.605600, -1.200000, 0.355700),
    (0.598800, -1.243700, 0.351700),
    (0.692100, -1.243700, 0.000000),
    (0.716200, -1.243700, 0.000000),
    (0.716200, -1.243700, 0.000000),
    (0.619600, -1.243700, 0.363900),
    (0.598800, -1.243700, 0.351700),
    (0.619600, -1.243700, 0.363900),
    (0.716200, -1.243700, 0.000000),
    (0.750000, -1.200000, 0.000000),
    (0.750000, -1.200000, 0.000000),
    (0.648900, -1.200000, 0.381100),
    (0.619600, -1.243700, 0.363900),
    (0.750000, -1.200000, 0.000000),
    (0.648900, -1.200000, -0.381100),
    (0.753000, -0.938900, -0.442300),
    (0.753000, -0.938900, -0.442300),
    (0.870400, -0.938900, 0.000000),
    (0.750000, -1.200000, 0.000000),
    (0.870400, -0.938900, 0.000000),
    (0.753000, -0.938900, -0.442300),
    (0.833100, -0.686100, -0.489300),
    (0.833100, -0.686100, -0.489300),
    (0.963000, -0.686100, 0.000000),
    (0.870400, -0.938900, 0.000000),
    (0.963000, -0.686100, 0.000000),
    (0.833100, -0.686100, -0.489300),
    (0.865200, -0.450000, -0.508100),
    (0.865200, -0.450000, -0.508100),
    (1.000000, -0.450000, 0.000000),
    (0.963000, -0.686100, 0.000000),
    (0.648900, -1.200000, -0.381100),
    (0.381100, -1.200000, -0.648900),
    (0.442300, -0.938900, -0.753000),
    (0.442300, -0.938900, -0.753000),
    (0.753000, -0.938900, -0.442300),
    (0.648900, -1.200000, -0.381100),
    (0.753000, -0.938900, -0.442300),
    (0.442300, -0.938900, -0.753000),
    (0.489300, -0.686100, -0.833100),
    (0.489300, -0.686100, -0.833100),
    (0.833100, -0.686100, -0.489300),
    (0.753000, -0.938900, -0.442300),
    (0.833100, -0.686100, -0.489300),
    (0.489300, -0.686100, -0.833100),
    (0.508100, -0.450000, -0.865200),
    (0.508100, -0.450000, -0.865200),
    (0.865200, -0.450000, -0.508100),
    (0.833100, -0.686100, -0.489300),
    (0.381100, -1.200000, -0.648900),
    (0.000000, -1.200000, -0.750000),
    (0.000000, -0.938900, -0.870400),
    (0.000000, -0.938900, -0.870400),
    (0.442300, -0.938900, -0.753000),
    (0.381100, -1.200000, -0.648900),
    (0.442300, -0.938900, -0.753000),
    (0.000000, -0.938900, -0.870400),
    (0.000000, -0.686100, -0.963000),
    (0.000000, -0.686100, -0.963000),
    (0.489300, -0.686100, -0.833100),
    (0.442300, -0.938900, -0.753000),
    (0.489300, -0.686100, -0.833100),
    (0.000000, -0.686100, -0.963000),
    (0.000000, -0.450000, -1.000000),
    (0.000000, -0.450000, -1.000000),
    (0.508100, -0.450000, -0.865200),
    (0.489300, -0.686100, -0.833100),
    (0.000000, -1.200000, -0.750000),
    (-0.381100, -1.200000, -0.648900),
    (-0.442300, -0.938900, -0.753000),
    (-0.442300, -0.938900, -0.753000),
    (0.000000, -0.938900, -0.870400),
    (0.000000, -1.200000, -0.750000),
    (0.000000, -0.938900, -0.870400),
    (-0.442300, -0.938900, -0.753000),
    (-0.489300, -0.686100, -0.833100),
    (-0.489300, -0.686100, -0.833100),
    (0.000000, -0.686100, -0.963000),
    (0.000000, -0.938900, -0.870400),
    (0.000000, -0.686100, -0.963000),
    (-0.489300, -0.686100, -0.833100),
    (-0.508100, -0.450000, -0.865200),
    (-0.508100, -0.450000, -0.865200),
    (0.000000, -0.450000, -1.000000),
    (0.000000, -0.686100, -0.963000),
    (-0.381100, -1.200000, -0.648900),
    (-0.648900, -1.200000, -0.381100),
    (-0.753000, -0.938900, -0.442300),
    (-0.753000, -0.938900, -0.442300),
    (-0.442300, -0.938900, -0.753000),
    (-0.381100, -1.200000, -0.648900),
    (-0.442300, -0.938900, -0.753000),
    (-0.753000, -0.938900, -0.442300),
    (-0.833100, -0.686100, -0.489300),
    (-0.833100, -0.686100, -0.489300),
    (-0.489300, -0.686100, -0.833100),
    (-0.442300, -0.938900, -0.753000),
    (-0.489300, -0.686100, -0.833100),
    (-0.833100, -0.686100, -0.489300),
    (-0.865200, -0.450000, -0.508100),
    (-0.865200, -0.450000, -0.508100),
    (-0.508100, -0.450000, -0.865200),
    (-0.489300, -0.686100, -0.833100),
    (-0.648900, -1.200000, -0.381100),
    (-0.750000, -1.200000, 0.000000),
    (-0.870400, -0.938900, 0.000000),
    (-0.870400, -0.938900, 0.000000),
    (-0.753000, -0.938900, -0.442300),
    (-0.648900, -1.200000, -0.381100),
    (-0.753000, -0.938900, -0.442300),
    (-0.870400, -0.938900, 0.000000),
    (-0.963000, -0.686100, 0.000000),
    (-0.963000, -0.686100, 0.000000),
    (-0.833100, -0.686100, -0.489300),
    (-0.753000, -0.938900, -0.442300),
    (-0.833100, -0.686100, -0.489300),
    (-0.963000, -0.686100, 0.000000),
    (-1.000000, -0.450000, 0.000000),
    (-1.000000, -0.450000, 0.000000),
    (-0.865200, -0.450000, -0.508100),
    (-0.833100, -0.686100, -0.489300),
    (-0.750000, -1.200000, 0.000000),
    (-0.648900, -1.200000, 0.381100),
    (-0.753000, -0.938900, 0.442300),
    (-0.753000, -0.938900, 0.442300),
    (-0.870400, -0.938900, 0.000000),
    (-0.750000, -1.200000, 0.000000),
    (-0.870400, -0.938900, 0.000000),
    (-0.753000, -0.938900, 0.442300),
    (-0.833100, -0.686100, 0.489300),
    (-0.833100, -0.686100, 0.489300),
    (-0.963000, -0.686100, 0.000000),
    (-0.870400, -0.938900, 0.000000),
    (-0.963000, -0.686100, 0.000000),
    (-0.833100, -0.686100, 0.489300),
    (-0.865200, -0.450000, 0.508100),
    (-0.865200, -0.450000, 0.508100),
    (-1.000000, -0.450000, 0.000000),
    (-0.963000, -0.686100, 0.000000),
    (-0.648900, -1.200000, 0.381100),
    (-0.381100, -1.200000, 0.648900),
    (-0.442300, -0.938900, 0.753000),
    (-0.442300, -0.938900, 0.753000),
    (-0.753000, -0.938900, 0.442300),
    (-0.648900, -1.200000, 0.381100),
    (-0.753000, -0.938900, 0.442300),
    (-0.442300, -0.938900, 0.753000),
    (-0.489300, -0.686100, 0.833100),
    (-0.489300, -0.686100, 0.833100),
    (-0.833100, -0.686100, 0.489300),
    (-0.753000, -0.938900, 0.442300),
    (-0.833100, -0.686100, 0.489300),
    (-0.489300, -0.686100, 0.833100),
    (-0.508100, -0.450000, 0.865200),
    (-0.508100, -0.450000, 0.865200),
    (-0.865200, -0.450000, 0.508100),
    (-0.833100, -0.686100, 0.489300),
    (-0.381100, -1.200000, 0.648900),
    (0.000000, -1.200000, 0.750000),
    (0.000000, -0.938900, 0.870400),
    (0.000000, -0.938900, 0.870400),
    (-0.442300, -0.938900, 0.753000),
    (-0.381100, -1.200000, 0.648900),
    (-0.442300, -0.938900, 0.753000),
    (0.000000, -0.938900, 0.870400),
    (0.000000, -0.686100, 0.963000),
    (0.000000, -0.686100, 0.963000),
    (-0.489300, -0.686100, 0.833100),
    (-0.442300, -0.938900, 0.753000),
    (-0.489300, -0.686100, 0.833100),
    (0.000000, -0.686100, 0.963000),
    (0.000000, -0.450000, 1.000000),
    (0.000000, -0.450000, 1.000000),
    (-0.508100, -0.450000, 0.865200),
    (-0.489300, -0.686100, 0.833100),
    (0.000000, -1.200000, 0.750000),
    (0.381100, -1.200000, 0.648900),
    (0.442300, -0.938900, 0.753000),
    (0.442300, -0.938900, 0.753000),
    (0.000000, -0.938900, 0.870400),
    (0.000000, -1.200000, 0.750000),
    (0.000000, -0.938900, 0.870400),
    (0.442300, -0.938900, 0.753000),
    (0.489300, -0.686100, 0.833100),
    (0.489300, -0.686100, 0.833100),
    (0.000000, -0.686100, 0.963000),
    (0.000000, -0.938900, 0.870400),
    (0.000000, -0.686100, 0.963000),
    (0.489300, -0.686100, 0.833100),
    (0.508100, -0.450000, 0.865200),
    (0.508100, -0.450000, 0.865200),
    (0.000000, -0.450000, 1.000000),
    (0.000000, -0.686100, 0.963000),
    (0.381100, -1.200000, 0.648900),
    (0.648900, -1.200000, 0.381100),
    (0.753000, -0.938900, 0.442300),
    (0.753000, -0.938900, 0.442300),
    (0.442300, -0.938900, 0.753000),
    (0.381100, -1.200000, 0.648900),
    (0.442300, -0.938900, 0.753000),
    (0.753000, -0.938900, 0.442300),
    (0.833100, -0.686100, 0.489300),
    (0.833100, -0.686100, 0.489300),
    (0.489300, -0.686100, 0.833100),
    (0.442300, -0.938900, 0.753000),
    (0.489300, -0.686100, 0.833100),
    (0.833100, -0.686100, 0.489300),
    (0.865200, -0.450000, 0.508100),
    (0.865200, -0.450000, 0.508100),
    (0.508100, -0.450000, 0.865200),
    (0.489300, -0.686100, 0.833100),
    (0.648900, -1.200000, 0.381100),
    (0.750000, -1.200000, 0.000000),
    (0.870400, -0.938900, 0.000000),
    (0.870400, -0.938900, 0.000000),
    (0.753000, -0.938900, 0.442300),
    (0.648900, -1.200000, 0.381100),
    (0.753000, -0.938900, 0.442300),
    (0.870400, -0.938900, 0.000000),
    (0.963000, -0.686100, 0.000000),
    (0.963000, -0.686100, 0.000000),
    (0.833100, -0.686100, 0.489300),
    (0.753000, -0.938900, 0.442300),
    (0.833100, -0.686100, 0.489300),
    (0.963000, -0.686100, 0.000000),
    (1.000000, -0.450000, 0.000000),
    (1.000000, -0.450000, 0.000000),
    (0.865200, -0.450000, 0.508100),
    (0.833100, -0.686100, 0.489300),
    (1.000000, -0.450000, 0.000000),
    (0.865200, -0.450000, -0.508100),
    (0.809100, -0.261100, -0.475200),
    (0.809100, -0.261100, -0.475200),
    (0.935200, -0.261100, 0.000000),
    (1.000000, -0.450000, 0.000000),
    (0.935200, -0.261100, 0.000000),
    (0.809100, -0.261100, -0.475200),
    (0.705000, -0.138900, -0.414000),
    (0.705000, -0.138900, -0.414000),
    (0.814800, -0.138900, 0.000000),
    (0.935200, -0.261100, 0.000000),
    (0.814800, -0.138900, 0.000000),
    (0.705000, -0.138900, -0.414000),
    (0.648900, -0.075000, -0.381100),
    (0.648900, -0.075000, -0.381100),
    (0.750000, -0.075000, 0.000000),
    (0.814800, -0.138900, 0.000000),
    (0.865200, -0.450000, -0.508100),
    (0.508100, -0.450000, -0.865200),
    (0.475200, -0.261100, -0.809100),
    (0.475200, -0.261100, -0.809100),
    (0.809100, -0.261100, -0.475200),
    (0.865200, -0.450000, -0.508100),
    (0.809100, -0.261100, -0.475200),
    (0.475200, -0.261100, -0.809100),
    (0.414000, -0.138900, -0.705000),
    (0.414000, -0.138900, -0.705000),
    (0.705000, -0.138900, -0.414000),
    (0.809100, -0.261100, -0.475200),
    (0.705000, -0.138900, -0.414000),
    (0.414000, -0.138900, -0.705000),
    (0.381100, -0.075000, -0.648900),
    (0.381100, -0.075000, -0.648900),
    (0.648900, -0.075000, -0.381100),
    (0.705000, -0.138900, -0.414000),
    (0.508100, -0.450000, -0.865200),
    (0.000000, -0.450000, -1.000000),
    (0.000000, -0.261100, -0.935200),
    (0.000000, -0.261100, -0.935200),
    (0.475200, -0.261100, -0.809100),
    (0.508100, -0.450000, -0.865200),
    (0.475200, -0.261100, -0.809100),
    (0.000000, -0.261100, -0.935200),
    (0.000000, -0.138900, -0.814800),
    (0.000000, -0.138900, -0.814800),
    (0.414000, -0.138900, -0.705000),
    (0.475200, -0.261100, -0.809100),
    (0.414000, -0.138900, -0.705000),
    (0.000000, -0.138900, -0.814800),
    (0.000000, -0.075000, -0.750000),
    (0.000000, -0.075000, -0.750000),
    (0.381100, -0.075000, -0.648900),
    (0.414000, -0.138900, -0.705000),
    (0.000000, -0.450000, -1.000000),
    (-0.508100, -0.450000, -0.865200),
    (-0.475200, -0.261100, -0.809100),
    (-0.475200, -0.261100, -0.809100),
    (0.000000, -0.261100, -0.935200),
    (0.000000, -0.450000, -1.000000),
    (0.000000, -0.261100, -0.935200),
    (-0.475200, -0.261100, -0.809100),
    (-0.414000, -0.138900, -0.705000),
    (-0.414000, -0.138900, -0.705000),
    (0.000000, -0.138900, -0.814800),
    (0.000000, -0.261100, -0.935200),
    (0.000000, -0.138900, -0.814800),
    (-0.414000, -0.138900, -0.705000),
    (-0.381100, -0.075000, -0.648900),
    (-0.381100, -0.075000, -0.648900),
    (0.000000, -0.075000, -0.750000),
    (0.000000, -0.138900, -0.814800),
    (-0.508100, -0.450000, -0.865200),
    (-0.865200, -0.450000, -0.508100),
    (-0.809100, -0.261100, -0.475200),
    (-0.809100, -0.261100, -0.475200),
    (-0.475200, -0.261100, -0.809100),
    (-0.508100, -0.450000, -0.865200),
    (-0.475200, -0.261100, -0.809100),
    (-0.809100, -0.261100, -0.475200),
    (-0.705000, -0.138900, -0.414000),
    (-0.705000, -0.138900, -0.414000),
    (-0.414000, -0.138900, -0.705000),
    (-0.475200, -0.261100, -0.809100),
    (-0.414000, -0.138900, -0.705000),
    (-0.705000, -0.138900, -0.414000),
    (-0.648900, -0.075000, -0.381100),
    (-0.648900, -0.075000, -0.381100),
    (-0.381100, -0.075000, -0.648900),
    (-0.414000, -0.138900, -0.705000),
    (-0.865200, -0.450000, -0.508100),
    (-1.000000, -0.450000, 0.000000),
    (-0.935200, -0.261100, 0.000000),
    (-0.935200, -0.261100, 0.000000),
    (-0.809100, -0.261100, -0.475200),
    (-0.865200, -0.450000, -0.508100),
    (-0.809100, -0.261100, -0.475200),
    (-0.935200, -0.261100, 0.000000),
    (-0.814800, -0.138900, 0.000000),
    (-0.814800, -0.138900, 0.000000),
    (-0.705000, -0.138900, -0.414000),
    (-0.809100, -0.261100, -0.475200),
    (-0.705000, -0.138900, -0.414000),
    (-0.814800, -0.138900, 0.000000),
    (-0.750000, -0.075000, 0.000000),
    (-0.750000, -0.075000, 0.000000),
    (-0.648900, -0.075000, -0.381100),
    (-0.705000, -0.138900, -0.414000),
    (-1.000000, -0.450000, 0.000000),
    (-0.865200, -0.450000, 0.508100),
    (-0.809100, -0.261100, 0.475200),
    (-0.809100, -0.261100, 0.475200),
    (-0.935200, -0.261100, 0.000000),
    (-1.000000, -0.450000, 0.000000),
    (-0.935200, -0.261100, 0.000000),
    (-0.809100, -0.261100, 0.475200),
    (-0.705000, -0.138900, 0.414000),
    (-0.705000, -0.138900, 0.414000),
    (-0.814800, -0.138900, 0.000000),
    (-0.935200, -0.261100, 0.000000),
    (-0.814800, -0.138900, 0.000000),
    (-0.705000, -0.138900, 0.414000),
    (-0.648900, -0.075000, 0.381100),
    (-0.648900, -0.075000, 0.381100),
    (-0.750000, -0.075000, 0.000000),
    (-0.814800, -0.138900, 0.000000),
    (-0.865200, -0.450000, 0.508100),
    (-0.508100, -0.450000, 0.865200),
    (-0.475200, -0.261100, 0.809100),
    (-0.475200, -0.261100, 0.809100),
    (-0.809100, -0.261100, 0.475200),
    (-0.865200, -0.450000, 0.508100),
    (-0.809100, -0.261100, 0.475200),
    (-0.475200, -0.261100, 0.809100),
    (-0.414000, -0.138900, 0.705000),
    (-0.414000, -0.138900, 0.705000),
    (-0.705000, -0.138900, 0.414000),
    (-0.809100, -0.261100, 0.475200),
    (-0.705000, -0.138900, 0.414000),
    (-0.414000, -0.138900, 0.705000),
    (-0.381100, -0.075000, 0.648900),
    (-0.381100, -0.075000, 0.648900),
    (-0.648900, -0.075000, 0.381100),
    (-0.705000, -0.138900, 0.414000),
    (-0.508100, -0.450000, 0.865200),
    (0.000000, -0.450000, 1.000000),
    (0.000000, -0.261100, 0.935200),
    (0.000000, -0.261100, 0.935200),
    (-0.475200, -0.261100, 0.809100),
    (-0.508100, -0.450000, 0.865200),
    (-0.475200, -0.261100, 0.809100),
    (0.000000, -0.261100, 0.935200),
    (0.000000, -0.138900, 0.814800),
    (0.000000, -0.138900, 0.814800),
    (-0.414000, -0.138900, 0.705000),
    (-0.475200, -0.261100, 0.809100),
    (-0.414000, -0.138900, 0.705000),
    (0.000000, -0.138900, 0.814800),
    (0.000000, -0.075000, 0.750000),
    (0.000000, -0.075000, 0.750000),
    (-0.381100, -0.075000, 0.648900),
    (-0.414000, -0.138900, 0.705000),
    (0.000000, -0.450000, 1.000000),
    (0.508100, -0.450000, 0.865200),
    (0.475200, -0.261100, 0.809100),
    (0.475200, -0.261100, 0.809100),
    (0.000000, -0.261100, 0.935200),
    (0.000000, -0.450000, 1.000000),
    (0.000000, -0.261100, 0.935200),
    (0.475200, -0.261100, 0.809100),
    (0.414000, -0.138900, 0.705000),
    (0.414000, -0.138900, 0.705000),
    (0.000000, -0.138900, 0.814800),
    (0.000000, -0.261100, 0.935200),
    (0.000000, -0.138900, 0.814800),
    (0.414000, -0.138900, 0.705000),
    (0.381100, -0.075000, 0.648900),
    (0.381100, -0.075000, 0.648900),
    (0.000000, -0.075000, 0.750000),
    (0.000000, -0.138900, 0.814800),
    (0.508100, -0.450000, 0.865200),
    (0.865200, -0.450000, 0.508100),
    (0.809100, -0.261100, 0.475200),
    (0.809100, -0.261100, 0.475200),
    (0.475200, -0.261100, 0.809100),
    (0.508100, -0.450000, 0.865200),
    (0.475200, -0.261100, 0.809100),
    (0.809100, -0.261100, 0.475200),
    (0.705000, -0.138900, 0.414000),
    (0.705000, -0.138900, 0.414000),
    (0.414000, -0.138900, 0.705000),
    (0.475200, -0.261100, 0.809100),
    (0.414000, -0.138900, 0.705000),
    (0.705000, -0.138900, 0.414000),
    (0.648900, -0.075000, 0.381100),
    (0.648900, -0.075000, 0.381100),
    (0.381100, -0.075000, 0.648900),
    (0.414000, -0.138900, 0.705000),
    (0.865200, -0.450000, 0.508100),
    (1.000000, -0.450000, 0.000000),
    (0.935200, -0.261100, 0.000000),
    (0.935200, -0.261100, 0.000000),
    (0.809100, -0.261100, 0.475200),
    (0.865200, -0.450000, 0.508100),
    (0.809100, -0.261100, 0.475200),
    (0.935200, -0.261100, 0.000000),
    (0.814800, -0.138900, 0.000000),
    (0.814800, -0.138900, 0.000000),
    (0.705000, -0.138900, 0.414000),
    (0.809100, -0.261100, 0.475200),
    (0.705000, -0.138900, 0.414000),
    (0.814800, -0.138900, 0.000000),
    (0.750000, -0.075000, 0.000000),
    (0.750000, -0.075000, 0.000000),
    (0.648900, -0.075000, 0.381100),
    (0.705000, -0.138900, 0.414000),
    (0.750000, -0.075000, 0.000000),
    (0.648900, -0.075000, -0.381100),
    (0.617600, -0.038900, -0.362800),
    (0.617600, -0.038900, -0.362800),
    (0.713900, -0.038900, 0.000000),
    (0.750000, -0.075000, 0.000000),
    (0.713900, -0.038900, 0.000000),
    (0.617600, -0.038900, -0.362800),
    (0.442200, -0.011100, -0.259700),
    (0.442200, -0.011100, -0.259700),
    (0.511100, -0.011100, 0.000000),
    (0.713900, -0.038900, 0.000000),
    (0.511100, -0.011100, 0.000000),
    (0.442200, -0.011100, -0.259700),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.511100, -0.011100, 0.000000),
    (0.648900, -0.075000, -0.381100),
    (0.381100, -0.075000, -0.648900),
    (0.362800, -0.038900, -0.617600),
    (0.362800, -0.038900, -0.617600),
    (0.617600, -0.038900, -0.362800),
    (0.648900, -0.075000, -0.381100),
    (0.617600, -0.038900, -0.362800),
    (0.362800, -0.038900, -0.617600),
    (0.259700, -0.011100, -0.442200),
    (0.259700, -0.011100, -0.442200),
    (0.442200, -0.011100, -0.259700),
    (0.617600, -0.038900, -0.362800),
    (0.442200, -0.011100, -0.259700),
    (0.259700, -0.011100, -0.442200),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.442200, -0.011100, -0.259700),
    (0.381100, -0.075000, -0.648900),
    (0.000000, -0.075000, -0.750000),
    (0.000000, -0.038900, -0.713900),
    (0.000000, -0.038900, -0.713900),
    (0.362800, -0.038900, -0.617600),
    (0.381100, -0.075000, -0.648900),
    (0.362800, -0.038900, -0.617600),
    (0.000000, -0.038900, -0.713900),
    (0.000000, -0.011100, -0.511100),
    (0.000000, -0.011100, -0.511100),
    (0.259700, -0.011100, -0.442200),
    (0.362800, -0.038900, -0.617600),
    (0.259700, -0.011100, -0.442200),
    (0.000000, -0.011100, -0.511100),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.259700, -0.011100, -0.442200),
    (0.000000, -0.075000, -0.750000),
    (-0.381100, -0.075000, -0.648900),
    (-0.362800, -0.038900, -0.617600),
    (-0.362800, -0.038900, -0.617600),
    (0.000000, -0.038900, -0.713900),
    (0.000000, -0.075000, -0.750000),
    (0.000000, -0.038900, -0.713900),
    (-0.362800, -0.038900, -0.617600),
    (-0.259700, -0.011100, -0.442200),
    (-0.259700, -0.011100, -0.442200),
    (0.000000, -0.011100, -0.511100),
    (0.000000, -0.038900, -0.713900),
    (0.000000, -0.011100, -0.511100),
    (-0.259700, -0.011100, -0.442200),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, -0.011100, -0.511100),
    (-0.381100, -0.075000, -0.648900),
    (-0.648900, -0.075000, -0.381100),
    (-0.617600, -0.038900, -0.362800),
    (-0.617600, -0.038900, -0.362800),
    (-0.362800, -0.038900, -0.617600),
    (-0.381100, -0.075000, -0.648900),
    (-0.362800, -0.038900, -0.617600),
    (-0.617600, -0.038900, -0.362800),
    (-0.442200, -0.011100, -0.259700),
    (-0.442200, -0.011100, -0.259700),
    (-0.259700, -0.011100, -0.442200),
    (-0.362800, -0.038900, -0.617600),
    (-0.259700, -0.011100, -0.442200),
    (-0.442200, -0.011100, -0.259700),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (-0.259700, -0.011100, -0.442200),
    (-0.648900, -0.075000, -0.381100),
    (-0.750000, -0.075000, 0.000000),
    (-0.713900, -0.038900, 0.000000),
    (-0.713900, -0.038900, 0.000000),
    (-0.617600, -0.038900, -0.362800),
    (-0.648900, -0.075000, -0.381100),
    (-0.617600, -0.038900, -0.362800),
    (-0.713900, -0.038900, 0.000000),
    (-0.511100, -0.011100, 0.000000),
    (-0.511100, -0.011100, 0.000000),
    (-0.442200, -0.011100, -0.259700),
    (-0.617600, -0.038900, -0.362800),
    (-0.442200, -0.011100, -0.259700),
    (-0.511100, -0.011100, 0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (-0.442200, -0.011100, -0.259700),
    (-0.750000, -0.075000, 0.000000),
    (-0.648900, -0.075000, 0.381100),
    (-0.617600, -0.038900, 0.362800),
    (-0.617600, -0.038900, 0.362800),
    (-0.713900, -0.038900, 0.000000),
    (-0.750000, -0.075000, 0.000000),
    (-0.713900, -0.038900, 0.000000),
    (-0.617600, -0.038900, 0.362800),
    (-0.442200, -0.011100, 0.259700),
    (-0.442200, -0.011100, 0.259700),
    (-0.511100, -0.011100, 0.000000),
    (-0.713900, -0.038900, 0.000000),
    (-0.511100, -0.011100, 0.000000),
    (-0.442200, -0.011100, 0.259700),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (-0.511100, -0.011100, 0.000000),
    (-0.648900, -0.075000, 0.381100),
    (-0.381100, -0.075000, 0.648900),
    (-0.362800, -0.038900, 0.617600),
    (-0.362800, -0.038900, 0.617600),
    (-0.617600, -0.038900, 0.362800),
    (-0.648900, -0.075000, 0.381100),
    (-0.617600, -0.038900, 0.362800),
    (-0.362800, -0.038900, 0.617600),
    (-0.259700, -0.011100, 0.442200),
    (-0.259700, -0.011100, 0.442200),
    (-0.442200, -0.011100, 0.259700),
    (-0.617600, -0.038900, 0.362800),
    (-0.442200, -0.011100, 0.259700),
    (-0.259700, -0.011100, 0.442200),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (-0.442200, -0.011100, 0.259700),
    (-0.381100, -0.075000, 0.648900),
    (0.000000, -0.075000, 0.750000),
    (0.000000, -0.038900, 0.713900),
    (0.000000, -0.038900, 0.713900),
    (-0.362800, -0.038900, 0.617600),
    (-0.381100, -0.075000, 0.648900),
    (-0.362800, -0.038900, 0.617600),
    (0.000000, -0.038900, 0.713900),
    (0.000000, -0.011100, 0.511100),
    (0.000000, -0.011100, 0.511100),
    (-0.259700, -0.011100, 0.442200),
    (-0.362800, -0.038900, 0.617600),
    (-0.259700, -0.011100, 0.442200),
    (0.000000, -0.011100, 0.511100),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (-0.259700, -0.011100, 0.442200),
    (0.000000, -0.075000, 0.750000),
    (0.381100, -0.075000, 0.648900),
    (0.362800, -0.038900, 0.617600),
    (0.362800, -0.038900, 0.617600),
    (0.000000, -0.038900, 0.713900),
    (0.000000, -0.075000, 0.750000),
    (0.000000, -0.038900, 0.713900),
    (0.362800, -0.038900, 0.617600),
    (0.259700, -0.011100, 0.442200),
    (0.259700, -0.011100, 0.442200),
    (0.000000, -0.011100, 0.511100),
    (0.000000, -0.038900, 0.713900),
    (0.000000, -0.011100, 0.511100),
    (0.259700, -0.011100, 0.442200),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, -0.011100, 0.511100),
    (0.381100, -0.075000, 0.648900),
    (0.648900, -0.075000, 0.381100),
    (0.617600, -0.038900, 0.362800),
    (0.617600, -0.038900, 0.362800),
    (0.362800, -0.038900, 0.617600),
    (0.381100, -0.075000, 0.648900),
    (0.362800, -0.038900, 0.617600),
    (0.617600, -0.038900, 0.362800),
    (0.442200, -0.011100, 0.259700),
    (0.442200, -0.011100, 0.259700),
    (0.259700, -0.011100, 0.442200),
    (0.362800, -0.038900, 0.617600),
    (0.259700, -0.011100, 0.442200),
    (0.442200, -0.011100, 0.259700),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.259700, -0.011100, 0.442200),
    (0.648900, -0.075000, 0.381100),
    (0.750000, -0.075000, 0.000000),
    (0.713900, -0.038900, 0.000000),
    (0.713900, -0.038900, 0.000000),
    (0.617600, -0.038900, 0.362800),
    (0.648900, -0.075000, 0.381100),
    (0.617600, -0.038900, 0.362800),
    (0.713900, -0.038900, 0.000000),
    (0.511100, -0.011100, 0.000000),
    (0.511100, -0.011100, 0.000000),
    (0.442200, -0.011100, 0.259700),
    (0.617600, -0.038900, 0.362800),
    (0.442200, -0.011100, 0.259700),
    (0.511100, -0.011100, 0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.000000, 0.000000, -0.000000),
    (0.442200, -0.011100, 0.259700),
    (-0.800000, -1.012500, 0.000000),
    (-0.787000, -1.041700, -0.100000),
    (-1.115900, -1.036400, -0.100000),
    (-1.115900, -1.036400, -0.100000),
    (-1.098100, -1.008300, 0.000000),
    (-0.800000, -1.012500, 0.000000),
    (-1.098100, -1.008300, 0.000000),
    (-1.115900, -1.036400, -0.100000),
    (-1.319300, -0.999700, -0.100000),
    (-1.319300, -0.999700, -0.100000),
    (-1.285200, -0.979200, 0.000000),
    (-1.098100, -1.008300, 0.000000),
    (-1.285200, -0.979200, 0.000000),
    (-1.319300, -0.999700, -0.100000),
    (-1.388900, -0.900000, -0.100000),
    (-1.388900, -0.900000, -0.100000),
    (-1.350000, -0.900000, 0.000000),
    (-1.285200, -0.979200, 0.000000),
    (-0.787000, -1.041700, -0.100000),
    (-0.763000, -1.095800, -0.100000),
    (-1.148900, -1.088600, -0.100000),
    (-1.148900, -1.088600, -0.100000),
    (-1.115900, -1.036400, -0.100000),
    (-0.787000, -1.041700, -0.100000),
    (-1.115900, -1.036400, -0.100000),
    (-1.148900, -1.088600, -0.100000),
    (-1.382600, -1.037800, -0.100000),
    (-1.382600, -1.037800, -0.100000),
    (-1.319300, -0.999700, -0.100000),
    (-1.115900, -1.036400, -0.100000),
    (-1.319300, -0.999700, -0.100000),
    (-1.382600, -1.037800, -0.100000),
    (-1.461100, -0.900000, -0.100000),
    (-1.461100, -0.900000, -0.100000),
    (-1.388900, -0.900000, -0.100000),
    (-1.319300, -0.999700, -0.100000),
    (-0.763000, -1.095800, -0.100000),
    (-0.750000, -1.125000, 0.000000),
    (-1.166700, -1.116700, 0.000000),
    (-1.166700, -1.116700, 0.000000),
    (-1.148900, -1.088600, -0.100000),
    (-0.763000, -1.095800, -0.100000),
    (-1.148900, -1.088600, -0.100000),
    (-1.166700, -1.116700, 0.000000),
    (-1.416700, -1.058300, 0.000000),
    (-1.416700, -1.058300, 0.000000),
    (-1.382600, -1.037800, -0.100000),
    (-1.148900, -1.088600, -0.100000),
    (-1.382600, -1.037800, -0.100000),
    (-1.416700, -1.058300, 0.000000),
    (-1.500000, -0.900000, 0.000000),
    (-1.500000, -0.900000, 0.000000),
    (-1.461100, -0.900000, -0.100000),
    (-1.382600, -1.037800, -0.100000),
    (-0.750000, -1.125000, 0.000000),
    (-0.763000, -1.095800, 0.100000),
    (-1.148900, -1.088600, 0.100000),
    (-1.148900, -1.088600, 0.100000),
    (-1.166700, -1.116700, 0.000000),
    (-0.750000, -1.125000, 0.000000),
    (-1.166700, -1.116700, 0.000000),
    (-1.148900, -1.088600, 0.100000),
    (-1.382600, -1.037800, 0.100000),
    (-1.382600, -1.037800, 0.100000),
    (-1.416700, -1.058300, 0.000000),
    (-1.166700, -1.116700, 0.000000),
    (-1.416700, -1.058300, 0.000000),
    (-1.382600, -1.037800, 0.100000),
    (-1.461100, -0.900000, 0.100000),
    (-1.461100, -0.900000, 0.100000),
    (-1.500000, -0.900000, 0.000000),
    (-1.416700, -1.058300, 0.000000),
    (-0.763000, -1.095800, 0.100000),
    (-0.787000, -1.041700, 0.100000),
    (-1.115900, -1.036400, 0.100000),
    (-1.115900, -1.036400, 0.100000),
    (-1.148900, -1.088600, 0.100000),
    (-0.763000, -1.095800, 0.100000),
    (-1.148900, -1.088600, 0.100000),
    (-1.115900, -1.036400, 0.100000),
    (-1.319300, -0.999700, 0.100000),
    (-1.319300, -0.999700, 0.100000),
    (-1.382600, -1.037800, 0.100000),
    (-1.148900, -1.088600, 0.100000),
    (-1.382600, -1.037800, 0.100000),
    (-1.319300, -0.999700, 0.100000),
    (-1.388900, -0.900000, 0.100000),
    (-1.388900, -0.900000, 0.100000),
    (-1.461100, -0.900000, 0.100000),
    (-1.382600, -1.037800, 0.100000),
    (-0.787000, -1.041700, 0.100000),
    (-0.800000, -1.012500, 0.000000),
    (-1.098100, -1.008300, 0.000000),
    (-1.098100, -1.008300, 0.000000),
    (-1.115900, -1.036400, 0.100000),
    (-0.787000, -1.041700, 0.100000),
    (-1.115900, -1.036400, 0.100000),
    (-1.098100, -1.008300, 0.000000),
    (-1.285200, -0.979200, 0.000000),
    (-1.285200, -0.979200, 0.000000),
    (-1.319300, -0.999700, 0.100000),
    (-1.115900, -1.036400, 0.100000),
    (-1.319300, -0.999700, 0.100000),
    (-1.285200, -0.979200, 0.000000),
    (-1.350000, -0.900000, 0.000000),
    (-1.350000, -0.900000, 0.000000),
    (-1.388900, -0.900000, 0.100000),
    (-1.319300, -0.999700, 0.100000),
    (-1.350000, -0.900000, 0.000000),
    (-1.388900, -0.900000, -0.100000),
    (-1.347500, -0.738500, -0.100000),
    (-1.347500, -0.738500, -0.100000),
    (-1.314800, -0.758300, 0.000000),
    (-1.350000, -0.900000, 0.000000),
    (-1.314800, -0.758300, 0.000000),
    (-1.347500, -0.738500, -0.100000),
    (-1.216700, -0.562900, -0.100000),
    (-1.216700, -0.562900, -0.100000),
    (-1.201900, -0.591700, 0.000000),
    (-1.314800, -0.758300, 0.000000),
    (-1.201900, -0.591700, 0.000000),
    (-1.216700, -0.562900, -0.100000),
    (-0.987000, -0.411100, -0.100000),
    (-0.987000, -0.411100, -0.100000),
    (-1.000000, -0.450000, 0.000000),
    (-1.201900, -0.591700, 0.000000),
    (-1.388900, -0.900000, -0.100000),
    (-1.461100, -0.900000, -0.100000),
    (-1.408100, -0.701700, -0.100000),
    (-1.408100, -0.701700, -0.100000),
    (-1.347500, -0.738500, -0.100000),
    (-1.388900, -0.900000, -0.100000),
    (-1.347500, -0.738500, -0.100000),
    (-1.408100, -0.701700, -0.100000),
    (-1.244400, -0.509400, -0.100000),
    (-1.244400, -0.509400, -0.100000),
    (-1.216700, -0.562900, -0.100000),
    (-1.347500, -0.738500, -0.100000),
    (-1.216700, -0.562900, -0.100000),
    (-1.244400, -0.509400, -0.100000),
    (-0.963000, -0.338900, -0.100000),
    (-0.963000, -0.338900, -0.100000),
    (-0.987000, -0.411100, -0.100000),
    (-1.216700, -0.562900, -0.100000),
    (-1.461100, -0.900000, -0.100000),
    (-1.500000, -0.900000, 0.000000),
    (-1.440700, -0.681900, 0.000000),
    (-1.440700, -0.681900, 0.000000),
    (-1.408100, -0.701700, -0.100000),
    (-1.461100, -0.900000, -0.100000),
    (-1.408100, -0.701700, -0.100000),
    (-1.440700, -0.681900, 0.000000),
    (-1.259300, -0.480600, 0.000000),
    (-1.259300, -0.480600, 0.000000),
    (-1.244400, -0.509400, -0.100000),
    (-1.408100, -0.701700, -0.100000),
    (-1.244400, -0.509400, -0.100000),
    (-1.259300, -0.480600, 0.000000),
    (-0.950000, -0.300000, 0.000000),
    (-0.950000, -0.300000, 0.000000),
    (-0.963000, -0.338900, -0.100000),
    (-1.244400, -0.509400, -0.100000),
    (-1.500000, -0.900000, 0.000000),
    (-1.461100, -0.900000, 0.100000),
    (-1.408100, -0.701700, 0.100000),
    (-1.408100, -0.701700, 0.100000),
    (-1.440700, -0.681900, 0.000000),
    (-1.500000, -0.900000, 0.000000),
    (-1.440700, -0.681900, 0.000000),
    (-1.408100, -0.701700, 0.100000),
    (-1.244400, -0.509400, 0.100000),
    (-1.244400, -0.509400, 0.100000),
    (-1.259300, -0.480600, 0.000000),
    (-1.440700, -0.681900, 0.000000),
    (-1.259300, -0.480600, 0.000000),
    (-1.244400, -0.509400, 0.100000),
    (-0.963000, -0.338900, 0.100000),
    (-0.963000, -0.338900, 0.100000),
    (-0.950000, -0.300000, 0.000000),
    (-1.259300, -0.480600, 0.000000),
    (-1.461100, -0.900000, 0.100000),
    (-1.388900, -0.900000, 0.100000),
    (-1.347500, -0.738500, 0.100000),
    (-1.347500, -0.738500, 0.100000),
    (-1.408100, -0.701700, 0.100000),
    (-1.461100, -0.900000, 0.100000),
    (-1.408100, -0.701700, 0.100000),
    (-1.347500, -0.738500, 0.100000),
    (-1.216700, -0.562900, 0.100000),
    (-1.216700, -0.562900, 0.100000),
    (-1.244400, -0.509400, 0.100000),
    (-1.408100, -0.701700, 0.100000),
    (-1.244400, -0.509400, 0.100000),
    (-1.216700, -0.562900, 0.100000),
    (-0.987000, -0.411100, 0.100000),
    (-0.987000, -0.411100, 0.100000),
    (-0.963000, -0.338900, 0.100000),
    (-1.244400, -0.509400, 0.100000),
    (-1.388900, -0.900000, 0.100000),
    (-1.350000, -0.900000, 0.000000),
    (-1.314800, -0.758300, 0.000000),
    (-1.314800, -0.758300, 0.000000),
    (-1.347500, -0.738500, 0.100000),
    (-1.388900, -0.900000, 0.100000),
    (-1.347500, -0.738500, 0.100000),
    (-1.314800, -0.758300, 0.000000),
    (-1.201900, -0.591700, 0.000000),
    (-1.201900, -0.591700, 0.000000),
    (-1.216700, -0.562900, 0.100000),
    (-1.347500, -0.738500, 0.100000),
    (-1.216700, -0.562900, 0.100000),
    (-1.201900, -0.591700, 0.000000),
    (-1.000000, -0.450000, 0.000000),
    (-1.000000, -0.450000, 0.000000),
    (-0.987000, -0.411100, 0.100000),
    (-1.216700, -0.562900, 0.100000),
    (0.850000, -0.712500, 0.000000),
    (0.850000, -0.605600, -0.220000),
    (1.169800, -0.737100, -0.184600),
    (1.169800, -0.737100, -0.184600),
    (1.135200, -0.805600, 0.000000),
    (0.850000, -0.712500, 0.000000),
    (1.135200, -0.805600, 0.000000),
    (1.169800, -0.737100, -0.184600),
    (1.274700, -0.981400, -0.118800),
    (1.274700, -0.981400, -0.118800),
    (1.231500, -1.006900, 0.000000),
    (1.135200, -0.805600, 0.000000),
    (1.231500, -1.006900, 0.000000),
    (1.274700, -0.981400, -0.118800),
    (1.427800, -1.200000, -0.083300),
    (1.427800, -1.200000, -0.083300),
    (1.350000, -1.200000, 0.000000),
    (1.231500, -1.006900, 0.000000),
    (0.850000, -0.605600, -0.220000),
    (0.850000, -0.406900, -0.220000),
    (1.234000, -0.610100, -0.184600),
    (1.234000, -0.610100, -0.184600),
    (1.169800, -0.737100, -0.184600),
    (0.850000, -0.605600, -0.220000),
    (1.169800, -0.737100, -0.184600),
    (1.234000, -0.610100, -0.184600),
    (1.354900, -0.933900, -0.118800),
    (1.354900, -0.933900, -0.118800),
    (1.274700, -0.981400, -0.118800),
    (1.169800, -0.737100, -0.184600),
    (1.274700, -0.981400, -0.118800),
    (1.354900, -0.933900, -0.118800),
    (1.572200, -1.200000, -0.083300),
    (1.572200, -1.200000, -0.083300),
    (1.427800, -1.200000, -0.083300),
    (1.274700, -0.981400, -0.118800),
    (0.850000, -0.406900, -0.220000),
    (0.850000, -0.300000, 0.000000),
    (1.268500, -0.541700, 0.000000),
    (1.268500, -0.541700, 0.000000),
    (1.234000, -0.610100, -0.184600),
    (0.850000, -0.406900, -0.220000),
    (1.234000, -0.610100, -0.184600),
    (1.268500, -0.541700, 0.000000),
    (1.398100, -0.908300, 0.000000),
    (1.398100, -0.908300, 0.000000),
    (1.354900, -0.933900, -0.118800),
    (1.234000, -0.610100, -0.184600),
    (1.354900, -0.933900, -0.118800),
    (1.398100, -0.908300, 0.000000),
    (1.650000, -1.200000, 0.000000),
    (1.650000, -1.200000, 0.000000),
    (1.572200, -1.200000, -0.083300),
    (1.354900, -0.933900, -0.118800),
    (0.850000, -0.300000, 0.000000),
    (0.850000, -0.406900, 0.220000),
    (1.234000, -0.610100, 0.184600),
    (1.234000, -0.610100, 0.184600),
    (1.268500, -0.541700, 0.000000),
    (0.850000, -0.300000, 0.000000),
    (1.268500, -0.541700, 0.000000),
    (1.234000, -0.610100, 0.184600),
    (1.354900, -0.933900, 0.118800),
    (1.354900, -0.933900, 0.118800),
    (1.398100, -0.908300, 0.000000),
    (1.268500, -0.541700, 0.000000),
    (1.398100, -0.908300, 0.000000),
    (1.354900, -0.933900, 0.118800),
    (1.572200, -1.200000, 0.083300),
    (1.572200, -1.200000, 0.083300),
    (1.650000, -1.200000, 0.000000),
    (1.398100, -0.908300, 0.000000),
    (0.850000, -0.406900, 0.220000),
    (0.850000, -0.605600, 0.220000),
    (1.169800, -0.737100, 0.184600),
    (1.169800, -0.737100, 0.184600),
    (1.234000, -0.610100, 0.184600),
    (0.850000, -0.406900, 0.220000),
    (1.234000, -0.610100, 0.184600),
    (1.169800, -0.737100, 0.184600),
    (1.274700, -0.981400, 0.118800),
    (1.274700, -0.981400, 0.118800),
    (1.354900, -0.933900, 0.118800),
    (1.234000, -0.610100, 0.184600),
    (1.354900, -0.933900, 0.118800),
    (1.274700, -0.981400, 0.118800),
    (1.427800, -1.200000, 0.083300),
    (1.427800, -1.200000, 0.083300),
    (1.572200, -1.200000, 0.083300),
    (1.354900, -0.933900, 0.118800),
    (0.850000, -0.605600, 0.220000),
    (0.850000, -0.712500, 0.000000),
    (1.135200, -0.805600, 0.000000),
    (1.135200, -0.805600, 0.000000),
    (1.169800, -0.737100, 0.184600),
    (0.850000, -0.605600, 0.220000),
    (1.169800, -0.737100, 0.184600),
    (1.135200, -0.805600, 0.000000),
    (1.231500, -1.006900, 0.000000),
    (1.231500, -1.006900, 0.000000),
    (1.274700, -0.981400, 0.118800),
    (1.169800, -0.737100, 0.184600),
    (1.274700, -0.981400, 0.118800),
    (1.231500, -1.006900, 0.000000),
    (1.350000, -1.200000, 0.000000),
    (1.350000, -1.200000, 0.000000),
    (1.427800, -1.200000, 0.083300),
    (1.274700, -0.981400, 0.118800),
    (1.350000, -1.200000, 0.000000),
    (1.427800, -1.200000, -0.083300),
    (1.478900, -1.227200, -0.074700),
    (1.478900, -1.227200, -0.074700),
    (1.396300, -1.225000, 0.000000),
    (1.350000, -1.200000, 0.000000),
    (1.396300, -1.225000, 0.000000),
    (1.478900, -1.227200, -0.074700),
    (1.491200, -1.227700, -0.058600),
    (1.491200, -1.227700, -0.058600),
    (1.420400, -1.225000, 0.000000),
    (1.396300, -1.225000, 0.000000),
    (1.420400, -1.225000, 0.000000),
    (1.491200, -1.227700, -0.058600),
    (1.451900, -1.200000, -0.050000),
    (1.451900, -1.200000, -0.050000),
    (1.400000, -1.200000, 0.000000),
    (1.420400, -1.225000, 0.000000),
    (1.427800, -1.200000, -0.083300),
    (1.572200, -1.200000, -0.083300),
    (1.632200, -1.231200, -0.074700),
    (1.632200, -1.231200, -0.074700),
    (1.478900, -1.227200, -0.074700),
    (1.427800, -1.200000, -0.083300),
    (1.478900, -1.227200, -0.074700),
    (1.632200, -1.231200, -0.074700),
    (1.622700, -1.232700, -0.058600),
    (1.622700, -1.232700, -0.058600),
    (1.491200, -1.227700, -0.058600),
    (1.478900, -1.227200, -0.074700),
    (1.491200, -1.227700, -0.058600),
    (1.622700, -1.232700, -0.058600),
    (1.548100, -1.200000, -0.050000),
    (1.548100, -1.200000, -0.050000),
    (1.451900, -1.200000, -0.050000),
    (1.491200, -1.227700, -0.058600),
    (1.572200, -1.200000, -0.083300),
    (1.650000, -1.200000, 0.000000),
    (1.714800, -1.233300, 0.000000),
    (1.714800, -1.233300, 0.000000),
    (1.632200, -1.231200, -0.074700),
    (1.572200, -1.200000, -0.083300),
    (1.632200, -1.231200, -0.074700),
    (1.714800, -1.233300, 0.000000),
    (1.693500, -1.235400, 0.000000),
    (1.693500, -1.235400, 0.000000),
    (1.622700, -1.232700, -0.058600),
    (1.632200, -1.231200, -0.074700),
    (1.622700, -1.232700, -0.058600),
    (1.693500, -1.235400, 0.000000),
    (1.600000, -1.200000, 0.000000),
    (1.600000, -1.200000, 0.000000),
    (1.548100, -1.200000, -0.050000),
    (1.622700, -1.232700, -0.058600),
    (1.650000, -1.200000, 0.000000),
    (1.572200, -1.200000, 0.083300),
    (1.632200, -1.231200, 0.074700),
    (1.632200, -1.231200, 0.074700),
    (1.714800, -1.233300, 0.000000),
    (1.650000, -1.200000, 0.000000),
    (1.714800, -1.233300, 0.000000),
    (1.632200, -1.231200, 0.074700),
    (1.622700, -1.232700, 0.058600),
    (1.622700, -1.232700, 0.058600),
    (1.693500, -1.235400, 0.000000),
    (1.714800, -1.233300, 0.000000),
    (1.693500, -1.235400, 0.000000),
    (1.622700, -1.232700, 0.058600),
    (1.548100, -1.200000, 0.050000),
    (1.548100, -1.200000, 0.050000),
    (1.600000, -1.200000, 0.000000),
    (1.693500, -1.235400, 0.000000),
    (1.572200, -1.200000, 0.083300),
    (1.427800, -1.200000, 0.083300),
    (1.478900, -1.227200, 0.074700),
    (1.478900, -1.227200, 0.074700),
    (1.632200, -1.231200, 0.074700),
    (1.572200, -1.200000, 0.083300),
    (1.632200, -1.231200, 0.074700),
    (1.478900, -1.227200, 0.074700),
    (1.491200, -1.227700, 0.058600),
    (1.491200, -1.227700, 0.058600),
    (1.622700, -1.232700, 0.058600),
    (1.632200, -1.231200, 0.074700),
    (1.622700, -1.232700, 0.058600),
    (1.491200, -1.227700, 0.058600),
    (1.451900, -1.200000, 0.050000),
    (1.451900, -1.200000, 0.050000),
    (1.548100, -1.200000, 0.050000),
    (1.622700, -1.232700, 0.058600),
    (1.427800, -1.200000, 0.083300),
    (1.350000, -1.200000, 0.000000),
    (1.396300, -1.225000, 0.000000),
    (1.396300, -1.225000, 0.000000),
    (1.478900, -1.227200, 0.074700),
    (1.427800, -1.200000, 0.083300),
    (1.478900, -1.227200, 0.074700),
    (1.396300, -1.225000, 0.000000),
    (1.420400, -1.225000, 0.000000),
    (1.420400, -1.225000, 0.000000),
    (1.491200, -1.227700, 0.058600),
    (1.478900, -1.227200, 0.074700),
    (1.491200, -1.227700, 0.058600),
    (1.420400, -1.225000, 0.000000),
    (1.400000, -1.200000, 0.000000),
    (1.400000, -1.200000, 0.000000),
    (1.451900, -1.200000, 0.050000),
    (1.491200, -1.227700, 0.058600),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.157100, -1.533300, -0.092400),
    (0.157100, -1.533300, -0.092400),
    (0.181500, -1.533300, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.181500, -1.533300, 0.000000),
    (0.157100, -1.533300, -0.092400),
    (0.102600, -1.441700, -0.060300),
    (0.102600, -1.441700, -0.060300),
    (0.118500, -1.441700, 0.000000),
    (0.181500, -1.533300, 0.000000),
    (0.118500, -1.441700, 0.000000),
    (0.102600, -1.441700, -0.060300),
    (0.086500, -1.350000, -0.050800),
    (0.086500, -1.350000, -0.050800),
    (0.100000, -1.350000, 0.000000),
    (0.118500, -1.441700, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.092400, -1.533300, -0.157100),
    (0.092400, -1.533300, -0.157100),
    (0.157100, -1.533300, -0.092400),
    (0.000000, -1.575000, 0.000000),
    (0.157100, -1.533300, -0.092400),
    (0.092400, -1.533300, -0.157100),
    (0.060300, -1.441700, -0.102600),
    (0.060300, -1.441700, -0.102600),
    (0.102600, -1.441700, -0.060300),
    (0.157100, -1.533300, -0.092400),
    (0.102600, -1.441700, -0.060300),
    (0.060300, -1.441700, -0.102600),
    (0.050800, -1.350000, -0.086500),
    (0.050800, -1.350000, -0.086500),
    (0.086500, -1.350000, -0.050800),
    (0.102600, -1.441700, -0.060300),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.533300, -0.181500),
    (0.000000, -1.533300, -0.181500),
    (0.092400, -1.533300, -0.157100),
    (0.000000, -1.575000, 0.000000),
    (0.092400, -1.533300, -0.157100),
    (0.000000, -1.533300, -0.181500),
    (0.000000, -1.441700, -0.118500),
    (0.000000, -1.441700, -0.118500),
    (0.060300, -1.441700, -0.102600),
    (0.092400, -1.533300, -0.157100),
    (0.060300, -1.441700, -0.102600),
    (0.000000, -1.441700, -0.118500),
    (0.000000, -1.350000, -0.100000),
    (0.000000, -1.350000, -0.100000),
    (0.050800, -1.350000, -0.086500),
    (0.060300, -1.441700, -0.102600),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.092400, -1.533300, -0.157100),
    (-0.092400, -1.533300, -0.157100),
    (0.000000, -1.533300, -0.181500),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.533300, -0.181500),
    (-0.092400, -1.533300, -0.157100),
    (-0.060300, -1.441700, -0.102600),
    (-0.060300, -1.441700, -0.102600),
    (0.000000, -1.441700, -0.118500),
    (0.000000, -1.533300, -0.181500),
    (0.000000, -1.441700, -0.118500),
    (-0.060300, -1.441700, -0.102600),
    (-0.050800, -1.350000, -0.086500),
    (-0.050800, -1.350000, -0.086500),
    (0.000000, -1.350000, -0.100000),
    (0.000000, -1.441700, -0.118500),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.157100, -1.533300, -0.092400),
    (-0.157100, -1.533300, -0.092400),
    (-0.092400, -1.533300, -0.157100),
    (0.000000, -1.575000, 0.000000),
    (-0.092400, -1.533300, -0.157100),
    (-0.157100, -1.533300, -0.092400),
    (-0.102600, -1.441700, -0.060300),
    (-0.102600, -1.441700, -0.060300),
    (-0.060300, -1.441700, -0.102600),
    (-0.092400, -1.533300, -0.157100),
    (-0.060300, -1.441700, -0.102600),
    (-0.102600, -1.441700, -0.060300),
    (-0.086500, -1.350000, -0.050800),
    (-0.086500, -1.350000, -0.050800),
    (-0.050800, -1.350000, -0.086500),
    (-0.060300, -1.441700, -0.102600),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.181500, -1.533300, 0.000000),
    (-0.181500, -1.533300, 0.000000),
    (-0.157100, -1.533300, -0.092400),
    (0.000000, -1.575000, 0.000000),
    (-0.157100, -1.533300, -0.092400),
    (-0.181500, -1.533300, 0.000000),
    (-0.118500, -1.441700, 0.000000),
    (-0.118500, -1.441700, 0.000000),
    (-0.102600, -1.441700, -0.060300),
    (-0.157100, -1.533300, -0.092400),
    (-0.102600, -1.441700, -0.060300),
    (-0.118500, -1.441700, 0.000000),
    (-0.100000, -1.350000, 0.000000),
    (-0.100000, -1.350000, 0.000000),
    (-0.086500, -1.350000, -0.050800),
    (-0.102600, -1.441700, -0.060300),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.157100, -1.533300, 0.092400),
    (-0.157100, -1.533300, 0.092400),
    (-0.181500, -1.533300, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.181500, -1.533300, 0.000000),
    (-0.157100, -1.533300, 0.092400),
    (-0.102600, -1.441700, 0.060300),
    (-0.102600, -1.441700, 0.060300),
    (-0.118500, -1.441700, 0.000000),
    (-0.181500, -1.533300, 0.000000),
    (-0.118500, -1.441700, 0.000000),
    (-0.102600, -1.441700, 0.060300),
    (-0.086500, -1.350000, 0.050800),
    (-0.086500, -1.350000, 0.050800),
    (-0.100000, -1.350000, 0.000000),
    (-0.118500, -1.441700, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (-0.092400, -1.533300, 0.157100),
    (-0.092400, -1.533300, 0.157100),
    (-0.157100, -1.533300, 0.092400),
    (0.000000, -1.575000, 0.000000),
    (-0.157100, -1.533300, 0.092400),
    (-0.092400, -1.533300, 0.157100),
    (-0.060300, -1.441700, 0.102600),
    (-0.060300, -1.441700, 0.102600),
    (-0.102600, -1.441700, 0.060300),
    (-0.157100, -1.533300, 0.092400),
    (-0.102600, -1.441700, 0.060300),
    (-0.060300, -1.441700, 0.102600),
    (-0.050800, -1.350000, 0.086500),
    (-0.050800, -1.350000, 0.086500),
    (-0.086500, -1.350000, 0.050800),
    (-0.102600, -1.441700, 0.060300),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.533300, 0.181500),
    (0.000000, -1.533300, 0.181500),
    (-0.092400, -1.533300, 0.157100),
    (0.000000, -1.575000, 0.000000),
    (-0.092400, -1.533300, 0.157100),
    (0.000000, -1.533300, 0.181500),
    (0.000000, -1.441700, 0.118500),
    (0.000000, -1.441700, 0.118500),
    (-0.060300, -1.441700, 0.102600),
    (-0.092400, -1.533300, 0.157100),
    (-0.060300, -1.441700, 0.102600),
    (0.000000, -1.441700, 0.118500),
    (0.000000, -1.350000, 0.100000),
    (0.000000, -1.350000, 0.100000),
    (-0.050800, -1.350000, 0.086500),
    (-0.060300, -1.441700, 0.102600),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.092400, -1.533300, 0.157100),
    (0.092400, -1.533300, 0.157100),
    (0.000000, -1.533300, 0.181500),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.533300, 0.181500),
    (0.092400, -1.533300, 0.157100),
    (0.060300, -1.441700, 0.102600),
    (0.060300, -1.441700, 0.102600),
    (0.000000, -1.441700, 0.118500),
    (0.000000, -1.533300, 0.181500),
    (0.000000, -1.441700, 0.118500),
    (0.060300, -1.441700, 0.102600),
    (0.050800, -1.350000, 0.086500),
    (0.050800, -1.350000, 0.086500),
    (0.000000, -1.350000, 0.100000),
    (0.000000, -1.441700, 0.118500),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.157100, -1.533300, 0.092400),
    (0.157100, -1.533300, 0.092400),
    (0.092400, -1.533300, 0.157100),
    (0.000000, -1.575000, 0.000000),
    (0.092400, -1.533300, 0.157100),
    (0.157100, -1.533300, 0.092400),
    (0.102600, -1.441700, 0.060300),
    (0.102600, -1.441700, 0.060300),
    (0.060300, -1.441700, 0.102600),
    (0.092400, -1.533300, 0.157100),
    (0.060300, -1.441700, 0.102600),
    (0.102600, -1.441700, 0.060300),
    (0.086500, -1.350000, 0.050800),
    (0.086500, -1.350000, 0.050800),
    (0.050800, -1.350000, 0.086500),
    (0.060300, -1.441700, 0.102600),
    (0.000000, -1.575000, 0.000000),
    (0.000000, -1.575000, 0.000000),
    (0.181500, -1.533300, 0.000000),
    (0.181500, -1.533300, 0.000000),
    (0.157100, -1.533300, 0.092400),
    (0.000000, -1.575000, 0.000000),
    (0.157100, -1.533300, 0.092400),
    (0.181500, -1.533300, 0.000000),
    (0.118500, -1.441700, 0.000000),
    (0.118500, -1.441700, 0.000000),
    (0.102600, -1.441700, 0.060300),
    (0.157100, -1.533300, 0.092400),
    (0.102600, -1.441700, 0.060300),
    (0.118500, -1.441700, 0.000000),
    (0.100000, -1.350000, 0.000000),
    (0.100000, -1.350000, 0.000000),
    (0.086500, -1.350000, 0.050800),
    (0.102600, -1.441700, 0.060300),
    (0.100000, -1.350000, 0.000000),
    (0.086500, -1.350000, -0.050800),
    (0.248300, -1.294400, -0.145900),
    (0.248300, -1.294400, -0.145900),
    (0.287000, -1.294400, 0.000000),
    (0.100000, -1.350000, 0.000000),
    (0.287000, -1.294400, 0.000000),
    (0.248300, -1.294400, -0.145900),
    (0.458200, -1.255600, -0.269100),
    (0.458200, -1.255600, -0.269100),
    (0.529600, -1.255600, 0.000000),
    (0.287000, -1.294400, 0.000000),
    (0.529600, -1.255600, 0.000000),
    (0.458200, -1.255600, -0.269100),
    (0.562400, -1.200000, -0.330300),
    (0.562400, -1.200000, -0.330300),
    (0.650000, -1.200000, 0.000000),
    (0.529600, -1.255600, 0.000000),
    (0.086500, -1.350000, -0.050800),
    (0.050800, -1.350000, -0.086500),
    (0.145900, -1.294400, -0.248300),
    (0.145900, -1.294400, -0.248300),
    (0.248300, -1.294400, -0.145900),
    (0.086500, -1.350000, -0.050800),
    (0.248300, -1.294400, -0.145900),
    (0.145900, -1.294400, -0.248300),
    (0.269100, -1.255600, -0.458200),
    (0.269100, -1.255600, -0.458200),
    (0.458200, -1.255600, -0.269100),
    (0.248300, -1.294400, -0.145900),
    (0.458200, -1.255600, -0.269100),
    (0.269100, -1.255600, -0.458200),
    (0.330300, -1.200000, -0.562400),
    (0.330300, -1.200000, -0.562400),
    (0.562400, -1.200000, -0.330300),
    (0.458200, -1.255600, -0.269100),
    (0.050800, -1.350000, -0.086500),
    (0.000000, -1.350000, -0.100000),
    (0.000000, -1.294400, -0.287000),
    (0.000000, -1.294400, -0.287000),
    (0.145900, -1.294400, -0.248300),
    (0.050800, -1.350000, -0.086500),
    (0.145900, -1.294400, -0.248300),
    (0.000000, -1.294400, -0.287000),
    (0.000000, -1.255600, -0.529600),
    (0.000000, -1.255600, -0.529600),
    (0.269100, -1.255600, -0.458200),
    (0.145900, -1.294400, -0.248300),
    (0.269100, -1.255600, -0.458200),
    (0.000000, -1.255600, -0.529600),
    (0.000000, -1.200000, -0.650000),
    (0.000000, -1.200000, -0.650000),
    (0.330300, -1.200000, -0.562400),
    (0.269100, -1.255600, -0.458200),
    (0.000000, -1.350000, -0.100000),
    (-0.050800, -1.350000, -0.086500),
    (-0.145900, -1.294400, -0.248300),
    (-0.145900, -1.294400, -0.248300),
    (0.000000, -1.294400, -0.287000),
    (0.000000, -1.350000, -0.100000),
    (0.000000, -1.294400, -0.287000),
    (-0.145900, -1.294400, -0.248300),
    (-0.269100, -1.255600, -0.458200),
    (-0.269100, -1.255600, -0.458200),
    (0.000000, -1.255600, -0.529600),
    (0.000000, -1.294400, -0.287000),
    (0.000000, -1.255600, -0.529600),
    (-0.269100, -1.255600, -0.458200),
    (-0.330300, -1.200000, -0.562400),
    (-0.330300, -1.200000, -0.562400),
    (0.000000, -1.200000, -0.650000),
    (0.000000, -1.255600, -0.529600),
    (-0.050800, -1.350000, -0.086500),
    (-0.086500, -1.350000, -0.050800),
    (-0.248300, -1.294400, -0.145900),
    (-0.248300, -1.294400, -0.145900),
    (-0.145900, -1.294400, -0.248300),
    (-0.050800, -1.350000, -0.086500),
    (-0.145900, -1.294400, -0.248300),
    (-0.248300, -1.294400, -0.145900),
    (-0.458200, -1.255600, -0.269100),
    (-0.458200, -1.255600, -0.269100),
    (-0.269100, -1.255600, -0.458200),
    (-0.145900, -1.294400, -0.248300),
    (-0.269100, -1.255600, -0.458200),
    (-0.458200, -1.255600, -0.269100),
    (-0.562400, -1.200000, -0.330300),
    (-0.562400, -1.200000, -0.330300),
    (-0.330300, -1.200000, -0.562400),
    (-0.269100, -1.255600, -0.458200),
    (-0.086500, -1.350000, -0.050800),
    (-0.100000, -1.350000, 0.000000),
    (-0.287000, -1.294400, 0.000000),
    (-0.287000, -1.294400, 0.000000),
    (-0.248300, -1.294400, -0.145900),
    (-0.086500, -1.350000, -0.050800),
    (-0.248300, -1.294400, -0.145900),
    (-0.287000, -1.294400, 0.000000),
    (-0.529600, -1.255600, 0.000000),
    (-0.529600, -1.255600, 0.000000),
    (-0.458200, -1.255600, -0.269100),
    (-0.248300, -1.294400, -0.145900),
    (-0.458200, -1.255600, -0.269100),
    (-0.529600, -1.255600, 0.000000),
    (-0.650000, -1.200000, 0.000000),
    (-0.650000, -1.200000, 0.000000),
    (-0.562400, -1.200000, -0.330300),
    (-0.458200, -1.255600, -0.269100),
    (-0.100000, -1.350000, 0.000000),
    (-0.086500, -1.350000, 0.050800),
    (-0.248300, -1.294400, 0.145900),
    (-0.248300, -1.294400, 0.145900),
    (-0.287000, -1.294400, 0.000000),
    (-0.100000, -1.350000, 0.000000),
    (-0.287000, -1.294400, 0.000000),
    (-0.248300, -1.294400, 0.145900),
    (-0.458200, -1.255600, 0.269100),
    (-0.458200, -1.255600, 0.269100),
    (-0.529600, -1.255600, 0.000000),
    (-0.287000, -1.294400, 0.000000),
    (-0.529600, -1.255600, 0.000000),
    (-0.458200, -1.255600, 0.269100),
    (-0.562400, -1.200000, 0.330300),
    (-0.562400, -1.200000, 0.330300),
    (-0.650000, -1.200000, 0.000000),
    (-0.529600, -1.255600, 0.000000),
    (-0.086500, -1.350000, 0.050800),
    (-0.050800, -1.350000, 0.086500),
    (-0.145900, -1.294400, 0.248300),
    (-0.145900, -1.294400, 0.248300),
    (-0.248300, -1.294400, 0.145900),
    (-0.086500, -1.350000, 0.050800),
    (-0.248300, -1.294400, 0.145900),
    (-0.145900, -1.294400, 0.248300),
    (-0.269100, -1.255600, 0.458200),
    (-0.269100, -1.255600, 0.458200),
    (-0.458200, -1.255600, 0.269100),
    (-0.248300, -1.294400, 0.145900),
    (-0.458200, -1.255600, 0.269100),
    (-0.269100, -1.255600, 0.458200),
    (-0.330300, -1.200000, 0.562400),
    (-0.330300, -1.200000, 0.562400),
    (-0.562400, -1.200000, 0.330300),
    (-0.458200, -1.255600, 0.269100),
    (-0.050800, -1.350000, 0.086500),
    (0.000000, -1.350000, 0.100000),
    (0.000000, -1.294400, 0.287000),
    (0.000000, -1.294400, 0.287000),
    (-0.145900, -1.294400, 0.248300),
    (-0.050800, -1.350000, 0.086500),
    (-0.145900, -1.294400, 0.248300),
    (0.000000, -1.294400, 0.287000),
    (0.000000, -1.255600, 0.529600),
    (0.000000, -1.255600, 0.529600),
    (-0.269100, -1.255600, 0.458200),
    (-0.145900, -1.294400, 0.248300),
    (-0.269100, -1.255600, 0.458200),
    (0.000000, -1.255600, 0.529600),
    (0.000000, -1.200000, 0.650000),
    (0.000000, -1.200000, 0.650000),
    (-0.330300, -1.200000, 0.562400),
    (-0.269100, -1.255600, 0.458200),
    (0.000000, -1.350000, 0.100000),
    (0.050800, -1.350000, 0.086500),
    (0.145900, -1.294400, 0.248300),
    (0.145900, -1.294400, 0.248300),
    (0.000000, -1.294400, 0.287000),
    (0.000000, -1.350000, 0.100000),
    (0.000000, -1.294400, 0.287000),
    (0.145900, -1.294400, 0.248300),
    (0.269100, -1.255600, 0.458200),
    (0.269100, -1.255600, 0.458200),
    (0.000000, -1.255600, 0.529600),
    (0.000000, -1.294400, 0.287000),
    (0.000000, -1.255600, 0.529600),
    (0.269100, -1.255600, 0.458200),
    (0.330300, -1.200000, 0.562400),
    (0.330300, -1.200000, 0.562400),
    (0.000000, -1.200000, 0.650000),
    (0.000000, -1.255600, 0.529600),
    (0.050800, -1.350000, 0.086500),
    (0.086500, -1.350000, 0.050800),
    (0.248300, -1.294400, 0.145900),
    (0.248300, -1.294400, 0.145900),
    (0.145900, -1.294400, 0.248300),
    (0.050800, -1.350000, 0.086500),
    (0.145900, -1.294400, 0.248300),
    (0.248300, -1.294400, 0.145900),
    (0.458200, -1.255600, 0.269100),
    (0.458200, -1.255600, 0.269100),
    (0.269100, -1.255600, 0.458200),
    (0.145900, -1.294400, 0.248300),
    (0.269100, -1.255600, 0.458200),
    (0.458200, -1.255600, 0.269100),
    (0.562400, -1.200000, 0.330300),
    (0.562400, -1.200000, 0.330300),
    (0.330300, -1.200000, 0.562400),
    (0.269100, -1.255600, 0.458200),
    (0.086500, -1.350000, 0.050800),
    (0.100000, -1.350000, 0.000000),
    (0.287000, -1.294400, 0.000000),
    (0.287000, -1.294400, 0.000000),
    (0.248300, -1.294400, 0.145900),
    (0.086500, -1.350000, 0.050800),
    (0.248300, -1.294400, 0.145900),
    (0.287000, -1.294400, 0.000000),
    (0.529600, -1.255600, 0.000000),
    (0.529600, -1.255600, 0.000000),
    (0.458200, -1.255600, 0.269100),
    (0.248300, -1.294400, 0.145900),
    (0.458200, -1.255600, 0.269100),
    (0.529600, -1.255600, 0.000000),
    (0.650000, -1.200000, 0.000000),
    (0.650000, -1.200000, 0.000000),
    (0.562400, -1.200000, 0.330300),
    (0.458200, -1.255600, 0.269100)
]

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar um slot (buffer).

In [13]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)


### Abaixo, nós enviamos todo o conteúdo da variável vertices.

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [14]:
# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Associando variáveis do programa GLSL (Vertex Shader) com nossos dados

Primeiro, definimos o byte inicial e o offset dos dados.

In [15]:
# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


Em seguida, soliciamos à GPU a localização da variável "position" (que guarda coordenadas dos nossos vértices). Nós definimos essa variável no Vertex Shader.

In [16]:
loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

A partir da localização anterior, nós indicamos à GPU onde está o conteúdo (via posições stride/offset) para a variável position (aqui identificada na posição loc).

Outros parâmetros:

* Definimos que possui três coordenadas
* Que cada coordenada é do tipo float (GL_FLOAT)
* Que não se deve normalizar a coordenada (False)

Mais detalhes: https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glVertexAttribPointer.xhtml

In [17]:
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Vamos pegar a localização da variável color para que possamos definir a cor em nosso laço da janela!

In [18]:
loc_color = glGetUniformLocation(program, "color")

### Eventos de Teclado

In [19]:
#variáveis para incrementos relativos à translação
t_x_inc = 0
t_y_inc = 0

#variáveis para incrementos relativos à rotação
angulo_x_inc = 0
angulo_y_inc = 0
angulo_z_inc = 0

#variáveis para incrementos relativos à escala uniforme
s_inc = 1.0

def key_event(window,key,scancode,action,mods):
    global t_x_inc, t_y_inc 
    global angulo_x_inc, angulo_y_inc, angulo_z_inc
    global s_inc

    if key == glfw.KEY_UP: t_y_inc += 0.01 # Isso faz o mesmo que #if key == 265: t_y_inc += 0.01 #cima
    if key == glfw.KEY_DOWN: t_y_inc -= 0.01
    if key == glfw.KEY_LEFT: t_x_inc -= 0.01 #esquerda
    if key == glfw.KEY_RIGHT: t_x_inc += 0.01 #direita

    
    if key == glfw.KEY_A: angulo_x_inc += 0.01 #rotacao x
    if key == glfw.KEY_S: angulo_y_inc += 0.01 #rotacao y
    if key == glfw.KEY_D: angulo_z_inc += 0.01 #rotacao z

    if key == glfw.KEY_Z: s_inc -= 0.01 #aumenta escala
    if key == glfw.KEY_X: s_inc += 0.01 #reduz escala


glfw.set_key_callback(window,key_event)

### Nesse momento, nós exibimos a janela!


In [20]:
glfw.show_window(window)

### Loop principal da janela.

In [21]:
anguloRotacao = 0.0

glEnable(GL_DEPTH_TEST) ### importante para 3D

def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c


while not glfw.window_should_close(window):

    #processa eventos pendentes na fila de eventos, como entrada do usuário (teclado, mouse, etc.) e eventos de janela (redimensionamento, fechamento, etc.)
    glfw.poll_events() 
    
    
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    
    glClearColor(1.0, 1.0, 1.0, 1.0)
    
    ### Translação
    
    t_x = t_x_inc
    t_y = t_y_inc
    t_z = 0.0
    
    mat_translate = np.array([      1.0,  0.0, 0.0, t_x, 
                                    0.0,  1.0, 0.0, t_y, 
                                    0.0,  0.0, 1.0, t_z, 
                                    0.0,  0.0, 0.0, 1.0], np.float32)

    ### Rotação
    
    angulo_x = angulo_x_inc
    angulo_y = angulo_y_inc
    angulo_z = angulo_z_inc
    
    mat_rotation_z = np.array([     math.cos(angulo_z), -math.sin(angulo_z), 0.0, 0.0, 
                                    math.sin(angulo_z),  math.cos(angulo_z), 0.0, 0.0, 
                                    0.0,      0.0, 1.0, 0.0, 
                                    0.0,      0.0, 0.0, 1.0], np.float32)
    
    mat_rotation_x = np.array([     1.0,   0.0,    0.0, 0.0, 
                                    0.0, math.cos(angulo_x), -math.sin(angulo_x), 0.0, 
                                    0.0, math.sin(angulo_x),  math.cos(angulo_x), 0.0, 
                                    0.0,   0.0,    0.0, 1.0], np.float32)
    
    mat_rotation_y = np.array([     math.cos(angulo_y),  0.0, math.sin(angulo_y), 0.0, 
                                    0.0,    1.0,   0.0, 0.0, 
                                    -math.sin(angulo_y), 0.0, math.cos(angulo_y), 0.0, 
                                    0.0,    0.0,   0.0, 1.0], np.float32)

    ### Escala (uniformed)
    s_x = s_inc
    s_y = s_inc
    s_z = s_inc
    mat_scale = np.array([          s_x,  0.0, 0.0, 0.0, 
                                    0.0,  s_y, 0.0, 0.0, 
                                    0.0,  0.0, s_z, 0.0, 
                                    0.0,  0.0, 0.0, 1.0], np.float32)
    

    ### Combina transformações
    mat_transform = multiplica_matriz(mat_translate,mat_rotation_x)
    mat_transform = multiplica_matriz(mat_rotation_z,mat_transform)
    mat_transform = multiplica_matriz(mat_rotation_y,mat_transform)
    mat_transform = multiplica_matriz(mat_scale,mat_transform)
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    #glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)

    #Colore triângulos
    for triangle in range(0,len(vertices),3):
       
        random.seed( triangle )
        R = random.random()
        G = random.random()
        B = random.random()  

        glUniform4f(loc_color, R, G, B, 1.0)
        glDrawArrays(GL_TRIANGLES, triangle, 3)    
    

    glfw.swap_buffers(window)

glfw.terminate()